# The following is a rough and messy notebook due to confusions in project but it contains processing of Qualifying of Sprint and Race

In [5]:
import pandas as pd
import numpy as np

In [6]:
quali_2025 = pd.read_csv('race_quali_2025.csv')
quali_2024 = pd.read_csv('race_quali_2024.csv')
quali_2023 = pd.read_csv('race_quali_2023.csv')
quali_2022 = pd.read_csv('race_quali_2022.csv')

In [103]:
quali_2022.to_csv("race_quali_2022.csv", index=False)
quali_2023.to_csv("race_quali_2023.csv", index=False)
quali_2024.to_csv("race_quali_2024.csv", index=False)
quali_2025.to_csv("race_quali_2025.csv", index=False)

In [118]:
sprint_quali_2021.to_csv("sprint_quali_2025.csv", index=False)

In [44]:
kaggle_sprint_res = pd.read_csv('filler-datasets/kaggle_sprint_res.csv')
kaggle_sprint_res.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,fastestLapTime,statusId
0,1,1061,830,9,33,2,1,1,1,3,17,25:38.426,1538426,14,1:30.013,1
1,2,1061,1,131,44,1,2,2,2,2,17,+1.430,1539856,17,1:29.937,1
2,3,1061,822,131,77,3,3,3,3,1,17,+7.502,1545928,17,1:29.958,1
3,4,1061,844,6,16,4,4,4,4,0,17,+11.278,1549704,16,1:30.163,1
4,5,1061,846,1,4,6,5,5,5,0,17,+24.111,1562537,16,1:30.566,1


## Adding RaceIds

### Checking the length for pointer assignment approach for assigning RaceIDs

In [57]:
races_kaggle = pd.read_csv('filler-datasets/races_filtered.csv')
kaggle_races = races_kaggle[races_kaggle['year']>2021]
kaggle_races.info()

<class 'pandas.core.frame.DataFrame'>
Index: 68 entries, 60 to 148
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   raceId     68 non-null     int64 
 1   year       68 non-null     int64 
 2   round      68 non-null     int64 
 3   circuitId  68 non-null     int64 
 4   name       68 non-null     object
dtypes: int64(4), object(1)
memory usage: 3.2+ KB


In [71]:
len(quali_2024['Track'].value_counts())+len(quali_2023['Track'].value_counts())+len(quali_2022['Track'].value_counts())

68

In [73]:
sprint_res_kaggle = pd.read_csv('filler-datasets/kaggle_sprint_res.csv')
len(sprint_res_kaggle['raceId'].value_counts())

18

In [108]:
len(sprint_quali_2024['Track'].value_counts())+len(sprint_quali_2023['Track'].value_counts())+len(sprint_quali_2021['Track'].value_counts())

18

In [125]:
sprint_quali_2023 = pd.read_csv('sprint_quali_2023.csv')
sprint_quali_2024 = pd.read_csv('sprint_quali_2024.csv')
sprint_quali_2025 = pd.read_csv('sprint_quali_2025.csv')

### Adding KaggleDriverID on the datasets

In [28]:
def normalize_name_series(s):
    return (
        s.str.lower()
         .str.strip()
         .str.normalize('NFKD')
         .str.encode('ascii', errors='ignore')
         .str.decode('utf-8')
    )

def add_kaggle_driver_id(df, kaggle_driver):
    df = df.copy()

    df['driver_norm'] = normalize_name_series(df['Driver'])

    kd = kaggle_driver.copy()
    kd['driver_norm'] = normalize_name_series(
        kd['forename'] + ' ' + kd['surname']
    )

    df = df.merge(
        kd[['driverId', 'driver_norm']],
        on='driver_norm',
        how='left'
    )

    df.rename(columns={'driverId': 'kaggle_driver_id'}, inplace=True)
    df.drop(columns=['driver_norm'], inplace=True)

    return df


In [38]:
quali_2022 = add_kaggle_driver_id(quali_2022, kaggle_driver)
#quali_2023 = add_kaggle_driver_id(quali_2023, kaggle_driver)
#quali_2024 = add_kaggle_driver_id(quali_2024, kaggle_driver)
#quali_2025 = add_kaggle_driver_id(quali_2025, kaggle_driver)

In [80]:
quali_2022['year'] = 2022
quali_2023['year'] = 2023
quali_2024['year'] = 2024

## Function to Add RaceID in Race Quali (It includes index checks too)

In [92]:
race_id_sequence = kaggle_races['raceId'].tolist()
def assign_race_ids_across_seasons(dfs, race_id_sequence):
    race_ptr = 0
    total_races = len(race_id_sequence)

    updated_dfs = []

    for df in dfs:
        df = df.copy()

        race_id_col = []
        tracks = df['Track'].values
        n = len(df)

        start = 0

        for i in range(1, n):
            if tracks[i] != tracks[i - 1]:
                if race_ptr >= total_races:
                    raise IndexError("Exceeded race_id_sequence length")

                race_id_col.extend(
                    [race_id_sequence[race_ptr]] * (i - start)
                )
                race_ptr += 1
                start = i

        # last block of this dataset
        if race_ptr >= total_races:
            raise IndexError("Exceeded race_id_sequence length")

        race_id_col.extend(
            [race_id_sequence[race_ptr]] * (n - start)
        )
        race_ptr += 1

        if len(race_id_col) != n:
            raise ValueError("raceId assignment length mismatch")

        df['raceId'] = race_id_col
        updated_dfs.append(df)

    # Final boundary checks
    if race_ptr != total_races:
        raise ValueError(
            f"RaceId mismatch: used {race_ptr} ids but Kaggle has {total_races}"
        )

    return updated_dfs

In [93]:
quali_2022, quali_2023, quali_2024 = assign_race_ids_across_seasons(
    [quali_2022, quali_2023, quali_2024],
    race_id_sequence
)

In [96]:
quali_2024.head()

,Track,Position,No,Driver,Team,Q1,Q2,Q3,Laps,kaggle_driver_id,year,raceId
0,Bahrain,1,1,Max Verstappen,Red Bull Racing Honda RBPT,1:30.031,1:29.374,1:29.179,17,830,2024,1121
1,Bahrain,2,16,Charles Leclerc,Ferrari,1:30.243,1:29.165,1:29.407,19,844,2024,1121
2,Bahrain,3,63,George Russell,Mercedes,1:30.350,1:29.922,1:29.485,17,847,2024,1121
3,Bahrain,4,55,Carlos Sainz,Ferrari,1:29.909,1:29.573,1:29.507,18,832,2024,1121
4,Bahrain,5,11,Sergio Perez,Red Bull Racing Honda RBPT,1:30.221,1:29.932,1:29.537,17,815,2024,1121


## Converting time to seconds and converting to final feature format

In [175]:
def quali_time_to_sec(val):
    if pd.isna(val):
        return np.nan

    val = str(val).strip()
    parts = val.split(':')

    try:
        if len(parts) == 2:
            # mm:ss.xxx
            m, s = parts
            return int(m) * 60 + float(s)
        elif len(parts) == 3:
            # hh:mm:ss.xxx (rare but exists)
            h, m, s = parts
            return int(h) * 3600 + int(m) * 60 + float(s)
    except:
        return np.nan

    return np.nan
def process_quali(df):
    df = df.copy()

    # participation flags
    df['participated_q2'] = df['Q2'].notna().astype(int)
    df['participated_q3'] = df['Q3'].notna().astype(int)

    # parse times
    df['q1_time_sec'] = df['Q1'].apply(quali_time_to_sec)
    df['q2_time_sec'] = df['Q2'].apply(quali_time_to_sec)
    df['q3_time_sec'] = df['Q3'].apply(quali_time_to_sec)

    # keep final columns
    df = df[
        [
            'raceId',
            'Track',
            'Team',
            'kaggle_driver_id',
            'Position',
            'participated_q2',
            'participated_q3',
            'q1_time_sec',
            'q2_time_sec',
            'q3_time_sec',
        ]
    ]

    return df


In [101]:
#quali_2022 = process_quali(quali_2022)
quali_2023 = process_quali(quali_2023)
quali_2024 = process_quali(quali_2024)

## Processing function for Sprint Qualifying

In [152]:
def sprint_quali_time_to_sec(val):
    if pd.isna(val):
        return np.nan

    val = str(val).strip()
    parts = val.split(':')

    try:
        if len(parts) == 2:
            # mm:ss.xxx
            m, s = parts
            return int(m) * 60 + float(s)
        elif len(parts) == 3:
            # hh:mm:ss.xxx (rare but exists)
            h, m, s = parts
            return int(h) * 3600 + int(m) * 60 + float(s)
    except:
        return np.nan

    return np.nan
def sprint_process_quali(df):
    df = df.copy()

    # participation flags
    df['participated_q2'] = df['SQ2'].notna().astype(int)
    df['participated_q3'] = df['SQ3'].notna().astype(int)

    # parse times
    df['q1_time_sec'] = df['SQ1'].apply(sprint_quali_time_to_sec)
    df['q2_time_sec'] = df['SQ2'].apply(sprint_quali_time_to_sec)
    df['q3_time_sec'] = df['SQ3'].apply(sprint_quali_time_to_sec)

    # keep final columns
    df = df[
        [
            'raceId',
            'Track',
            'Team',
            'kaggle_driver_id',
            'Position',
            'participated_q2',
            'participated_q3',
            'q1_time_sec',
            'q2_time_sec',
            'q3_time_sec',
        ]
    ]

    return df


In [133]:
sprint_quali_2023.head()

,Track,Position,No,Driver,Team,SQ1,SQ2,SQ3,Laps,kaggle_driver_id
0,Azerbaijan,1,16,Charles Leclerc,Ferrari,1:42.820,1:42.500,1:41.697,14,844
1,Azerbaijan,2,11,Sergio Perez,Red Bull Racing Honda RBPT,1:43.858,1:42.925,1:41.844,15,815
2,Azerbaijan,3,1,Max Verstappen,Red Bull Racing Honda RBPT,1:43.288,1:42.217,1:41.987,13,830
3,Azerbaijan,4,63,George Russell,Mercedes,1:43.763,1:43.112,1:42.252,18,847
4,Azerbaijan,5,55,Carlos Sainz,Ferrari,1:43.622,1:42.909,1:42.287,15,832


In [145]:
sprint_races_kaggle = pd.read_csv('filler-datasets/kaggle_sprint_res.csv')
sprint_raceId_from_2023 = sprint_races_kaggle[sprint_races_kaggle['raceId']>1095]['raceId'].unique().tolist()
#kaggle_races = races_kaggle[races_kaggle['year']>2021]
#kaggle_races.info()

## Since 2021 and 2022 dataset not there we assign raceId from 2023. 

In [146]:
sprint_raceId_from_2023

[1101, 1107, 1110, 1115, 1116, 1118, 1125, 1126, 1131, 1139, 1141, 1143]

In [147]:
def assign_sprint_race_ids(dfs, race_id_sequence):
    race_ptr = 0
    total_races = len(race_id_sequence)

    updated_dfs = []

    for df in dfs:
        df = df.copy()

        race_id_col = []
        tracks = df['Track'].values
        n = len(df)

        start = 0

        for i in range(1, n):
            if tracks[i] != tracks[i - 1]:
                if race_ptr >= total_races:
                    raise IndexError("Exceeded sprint raceId list length")

                race_id_col.extend(
                    [race_id_sequence[race_ptr]] * (i - start)
                )
                race_ptr += 1
                start = i

        # last block
        if race_ptr >= total_races:
            raise IndexError("Exceeded sprint raceId list length")

        race_id_col.extend(
            [race_id_sequence[race_ptr]] * (n - start)
        )
        race_ptr += 1

        if len(race_id_col) != n:
            raise ValueError("Sprint raceId assignment length mismatch")

        df['raceId'] = race_id_col
        updated_dfs.append(df)

    if race_ptr != total_races:
        raise ValueError(
            f"Sprint raceId mismatch: used {race_ptr} ids but list has {total_races}"
        )

    return updated_dfs

In [148]:
sprint_quali_2023, sprint_quali_2024 = assign_sprint_race_ids(
    [sprint_quali_2023, sprint_quali_2024],
    sprint_raceId_from_2023
)

In [160]:
sprint_quali_2024.head()

,Track,Position,No,Driver,Team,SQ1,SQ2,SQ3,Laps,kaggle_driver_id,raceId
0,China,1,4,Lando Norris,McLaren Mercedes,1:36.384,1:36.047,1:57.940,13,846,1125
1,China,2,44,Lewis Hamilton,Mercedes,1:37.181,1:36.287,1:59.201,15,1,1125
2,China,3,14,Fernando Alonso,Aston Martin Aramco Mercedes,1:36.883,1:36.119,1:59.915,14,4,1125
3,China,4,1,Max Verstappen,Red Bull Racing Honda RBPT,1:36.456,1:35.606,2:00.028,12,830,1125
4,China,5,55,Carlos Sainz,Ferrari,1:36.719,1:36.052,2:00.214,15,832,1125


In [154]:
sprint_quali_2024 = sprint_quali_2024.rename(columns={'Q1': 'SQ1'})
sprint_quali_2024 = sprint_quali_2024.rename(columns={'Q2': 'SQ2'})
sprint_quali_2024 = sprint_quali_2024.rename(columns={'Q3': 'SQ3'})

In [161]:
#sprint_quali_2023 = sprint_process_quali(sprint_quali_2023)
sprint_quali_2024 = sprint_process_quali(sprint_quali_2024)

In [159]:
sprint_quali_2023.head()

,raceId,Track,Team,kaggle_driver_id,Position,participated_q2,participated_q3,q1_time_sec,q2_time_sec,q3_time_sec
0,1101,Azerbaijan,Ferrari,844,1,1,1,102.820,102.500,101.697
1,1101,Azerbaijan,Red Bull Racing Honda RBPT,815,2,1,1,103.858,102.925,101.844
2,1101,Azerbaijan,Red Bull Racing Honda RBPT,830,3,1,1,103.288,102.217,101.987
3,1101,Azerbaijan,Mercedes,847,4,1,1,103.763,103.112,102.252
4,1101,Azerbaijan,Ferrari,832,5,1,1,103.622,102.909,102.287


In [166]:
sprint_quali_2024.head()

,raceId,Track,Team,kaggle_driver_id,Position,participated_q2,participated_q3,q1_time_sec,q2_time_sec,q3_time_sec
0,1125,China,McLaren Mercedes,846,1,1,1,96.384,96.047,117.940
1,1125,China,Mercedes,1,2,1,1,97.181,96.287,119.201
2,1125,China,Aston Martin Aramco Mercedes,4,3,1,1,96.883,96.119,119.915
3,1125,China,Red Bull Racing Honda RBPT,830,4,1,1,96.456,95.606,120.028
4,1125,China,Ferrari,832,5,1,1,96.719,96.052,120.214


In [164]:
sprint_quali_2024.to_csv("Quali-Final-Dataset/sprint_quali_2024.csv", index=False)
sprint_quali_2023.to_csv("Quali-Final-Dataset/sprint_quali_2023.csv", index=False)
sprint_quali_2025.to_csv("Quali-Final-Dataset/sprint_quali_2025.csv", index=False)

## Processing for 2025 Race Quali Data

In [168]:
race_quali_2025 = pd.read_csv('final_dataset(quali left to process)/race_quali_2025.csv')
race_quali_2025.head()

,Track,Position,No,Driver,Team,Q1,Q2,Q3,Laps,raceId,kaggle_driver_id
0,Australia,1,4,Lando Norris,McLaren Mercedes,1:15.912,1:15.415,1:15.096,20,1074,846
1,Australia,2,81,Oscar Piastri,McLaren Mercedes,1:16.062,1:15.468,1:15.180,18,1074,857
2,Australia,3,1,Max Verstappen,Red Bull Racing Honda RBPT,1:16.018,1:15.565,1:15.481,17,1074,830
3,Australia,4,63,George Russell,Mercedes,1:15.971,1:15.798,1:15.546,21,1074,847
4,Australia,5,22,Yuki Tsunoda,Racing Bulls Honda RBPT,1:16.225,1:16.009,1:15.670,18,1074,852


In [171]:
race_quali_2025 = race_quali_2025.drop('raceId',axis = 1)

In [174]:
race_quali_2025 = process_quali(race_quali_2025)

In [176]:
race_quali_2025.head()

,Track,Team,kaggle_driver_id,Position,participated_q2,participated_q3,q1_time_sec,q2_time_sec,q3_time_sec
0,Australia,McLaren Mercedes,846,1,1,1,75.912,75.415,75.096
1,Australia,McLaren Mercedes,857,2,1,1,76.062,75.468,75.180
2,Australia,Red Bull Racing Honda RBPT,830,3,1,1,76.018,75.565,75.481
3,Australia,Mercedes,847,4,1,1,75.971,75.798,75.546
4,Australia,Racing Bulls Honda RBPT,852,5,1,1,76.225,76.009,75.670


In [177]:
race_quali_2025.to_csv("Quali-Final-Dataset/race_quali_2025.csv", index=False)